# strategic_GNE_selection — independent reduction audit

This notebook uses the same **existing simulation root** as the demo, saves the 24h/1h reduced bilateral model in `fair_gne/1h/reduction/`, and reload-checks its full condensed matrices and audit. It is optional because the main demo rebuilds the reduction automatically for a new date.

## 0. Locate your existing study code

In [ ]:
from pathlib import Path
import sys
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE,*HERE.parents] if (p/'scenarios.py').is_file()),None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the ORIGINAL P2P repository (must find scenarios.py).')
DYNECT_REPO = Path.home()/'Documents/Github/dynect_py_p2p'
for folder in [ROOT, HERE, DYNECT_REPO]:
    if folder.is_dir() and str(folder) not in sys.path:
        sys.path.insert(0,str(folder))
import numpy as np
import pandas as pd
from IPython.display import display
from strategic_GNE_selection import (
    build_reduced_model,save_reduced_model,load_reduced_model,
    StrategicGame,model_max_violation,reduction_counts,fairness_metrics,
    save_candidate,certify_feeder,rebuild_constraints,
    scenario_paths,ensure_reduced_scenario,discover_prior_simulations)
print('Original repo:',ROOT)
print('Suite:',__import__('strategic_GNE_selection').__file__)
print('DyNECT:',__import__('dynect').__file__)
from p2p.setup import build_game,load_grid,evaluated_costs

## 1. Configure fidelity and reduction (no Julia enumeration)

In [ ]:
# ─── CHANGE THIS DATE AND RUN ALL ──────────────────────────────────────
DATE = '2016-06-15'  # e.g. '2016-06-15', '2016-01-07'
# ─────────────────────────────────────────────────────────────────────────
import config
STRUCTURE_NAME = config.structure_name()   # follows the config.py switch
N_PROSUMERS = config.PROSUMER_SCENARIO
SIMULATION_ROOT = None  # auto-find EXISTING results tree; override if ambiguous
SEED = 42
SOURCE_DT = 0.5
TARGET_DT = 1.0
CLEARING_EPSILON_KW = 0.0  # Exact bilateral reciprocity
INPUT_REGULARIZATION = 0.0
STATE_REGULARIZATION = 0.0
RUN_LOCAL_LP_PRUNING = True
LP_REDUNDANCY_TOL = 1e-9

# Set True when validating changed source data or reproducing reductions afresh.
FORCE_REBUILD = False
RUN_FULL_VGNE_FIDELITY_CHECK = True

SIMULATION_ROOT=find_simulation_root(ROOT,structure_name=STRUCTURE_NAME,
                                      override=SIMULATION_ROOT,cwd=HERE)
paths = scenario_paths(SIMULATION_ROOT, DATE, target_dt=TARGET_DT,
                       structure_name=STRUCTURE_NAME)
print('EXISTING simulation root:',SIMULATION_ROOT)
print('SAVE PROBE:',test_results_write_access(paths))
MODEL_PATH = paths['model_path']
AUDIT_PATH = paths['audit_path']
print('Scenario:', DATE)
print('Reduced snapshot:', MODEL_PATH)
print('Audit report:', AUDIT_PATH)

print('Original prior simulation files:',[r['kind'] for r in discover_prior_simulations(paths)])

## 2. Build or reload the correctly dated reduction

The helper compares date, seed, resolution, reduction options and suite implementation against the saved snapshot. A matching snapshot is reused; otherwise the full 1-hour scenario is reconstructed, certified, pruned, saved and reload-tested. Set `FORCE_REBUILD=True` if the underlying SimBench input files changed.


In [ ]:
import time
start = time.perf_counter()
reduced_1h, reduction_audit, scenario_info = ensure_reduced_scenario(
    DATE, base_dir=SIMULATION_ROOT, structure_name=STRUCTURE_NAME,
    seed=SEED, source_dt=SOURCE_DT,
    target_dt=TARGET_DT, lp_prune=RUN_LOCAL_LP_PRUNING,
    lp_tol=LP_REDUNDANCY_TOL,
    clearing_epsilon_kw=CLEARING_EPSILON_KW,
    input_regularization=INPUT_REGULARIZATION,
    state_regularization=STATE_REGULARIZATION,
    force_rebuild=FORCE_REBUILD,
)
assert reduction_audit['date'] == DATE
assert reduced_1h.N == N_PROSUMERS and reduced_1h.T == 24
assert float(reduced_1h.data['dt']) == TARGET_DT
print('Scenario source:', 'rebuilt' if scenario_info['rebuilt'] else 'cached')
print('Elapsed [s]:', round(time.perf_counter()-start, 2))
display(pd.DataFrame(reduction_audit['stages']))

## 3. Inspect *proofs* and exactly what was dropped

The feeder report distinguishes certified redundant rows from rows that must remain. `LP_unresolved` (here `unresolved`) counts LPs that were **not** used to delete inequalities. EV availability is aggregated using the conservative minimum over source slots; the number of mixed slots is reported as an approximation warning.

In [ ]:
feeder_table = pd.DataFrame(reduction_audit['certified_feeder'])
print('Feeder certificate results by type:')
display(feeder_table.groupby(['type','certified_redundant']).size().rename('rows').to_frame())
print('Retained shared inequality labels:')
print(reduced_1h.shared_labels)
print('Shared moved to local:', reduction_audit['moved_to_local'])
print('Mixed EV availability slots (aggregation warning):',
      reduction_audit['mixed_ev_availability_slots'])
if reduction_audit['local_lp_report']:
    lp_table = pd.DataFrame(reduction_audit['local_lp_report'])
    display(lp_table.groupby('agent')[['initial','remaining','removed','unresolved']].sum())
    assert (lp_table['unresolved'] >= 0).all()
print('Source vs target resolution:',SOURCE_DT,'->',TARGET_DT,'hours')


## 4. Confirm constraint and objective fidelity against the full **same-date, same-resolution** game

We reconstruct the full (unpruned) game using the saved scenario data. No second random daily scenario is generated. The comparison changes *only* the constraint reductions; it does not compare 1-hour temporal aggregation with the original half-hour model.


In [ ]:
u_full = None
u_reduced = None
if RUN_FULL_VGNE_FIDELITY_CHECK:
    import copy, gc
    from p2p.setup import build_game, evaluated_costs
    full_1h = build_game(copy.deepcopy(reduced_1h.data),
                         grid=load_grid(), **reduced_1h.choices)
    assert full_1h.N == reduced_1h.N and full_1h.T == reduced_1h.T
    assert all(label in full_1h.shared_labels
               for label in reduced_1h.shared_labels)
    start=time.perf_counter()
    original_solver=StrategicGame(full_1h)
    result_full=original_solver.solve_weighted()
    u_full=result_full['u']
    print('Full 1h solver [s]:',round(time.perf_counter()-start,2))
    del original_solver
    gc.collect()

    start=time.perf_counter()
    reduced_solver=StrategicGame(reduced_1h)
    result_reduced=reduced_solver.solve_weighted()
    u_reduced=result_reduced['u']
    print('Reduced 1h solver [s]:',round(time.perf_counter()-start,2))
    J_full=evaluated_costs(full_1h,u_full)
    J_reduced=evaluated_costs(reduced_1h,u_reduced)
    display(pd.DataFrame({'prosumer':np.arange(1,full_1h.N+1),
                          'full_cost_eur':J_full,'reduced_cost_eur':J_reduced,
                          'delta_eur':J_reduced-J_full}))
    print('Maximum |cost delta| EUR:',np.max(np.abs(J_reduced-J_full)))
    print('Total cost delta EUR:',np.sum(J_reduced)-np.sum(J_full))
    print('Max |dispatch delta| kW:',np.max(np.abs(u_full-u_reduced)))
    print('Reduced solution feasibility under FULL game:',
          model_max_violation(full_1h,u_reduced))
    print('Full solution feasibility under REDUCED game:',
          model_max_violation(reduced_1h,u_full))
    print('Cost-quad versus physical-cost difference [EUR]:',
          reduced_solver.model_cost_discrepancy(u_reduced))
    if np.max(np.abs(J_reduced-J_full))>1e-3:
        print('WARNING: Cost fidelity exceeds 1e-3 EUR. Investigate before selection.')
else:
    print('Full-vGNE fidelity check skipped by configuration.')


## 5. Verify the saved snapshot and audit for this date

`ensure_reduced_scenario()` already reloads and checks newly built `.npz` files. The following additionally verifies that both snapshot metadata and report are for the selected date.


In [ ]:
import json
saved_path = scenario_info['model_path']
AUDIT_PATH = scenario_info['audit_path']
assert saved_path.exists() and AUDIT_PATH.exists()
case_reloaded, audit_reloaded = load_reduced_model(saved_path,check_matrices=True)
assert audit_reloaded['date'] == DATE
assert case_reloaded.shared_labels == reduced_1h.shared_labels
assert json.loads(AUDIT_PATH.read_text())['date'] == DATE
print('Saved model:',saved_path,'size MB:',round(saved_path.stat().st_size/1e6,2))
print('Saved report:',AUDIT_PATH)
print('Reload SUCCESS:',reduction_counts(case_reloaded))
if u_reduced is not None:
    print('Saved model feasibility at reduced vGNE:',
          model_max_violation(case_reloaded,u_reduced))


## 6. Optional comparison to the original 30-minute 24h active-only model

**Disabled** by default: 48 half-hour periods increase the condensed problem. This additional comparison measures the *time-aggregation approximation*, not the structural feeder/local reductions. The original legacy active/reactive Gurobi formulation is still a separate model and would need its own validation.

In [ ]:
RUN_30MIN_COMPARISON = False
if RUN_30MIN_COMPARISON:
    from scenarios import make_scenario_horizon
    fine_data=make_scenario_horizon(DATE,seed=SEED,resolution_h=SOURCE_DT,lookahead_h=0)
    full_30min=build_game(fine_data,grid=load_grid(),
                          clearing_epsilon_kw=0.,input_regularization=0.,
                          state_regularization=0.)
    fine_sol=StrategicGame(full_30min).solve_weighted()
    J_30=evaluated_costs(full_30min,fine_sol['u'])
    print('24h / 30min total EUR:',J_30.sum())
    print('24h / 1h reduced total EUR:',evaluated_costs(reduced_1h,u_reduced).sum()
          if u_reduced is not None else 'run cell 4 first')
    print('Difference here includes aggregation, NOT just constraint screening.')
else:
    print('30-minute comparison disabled; enable if needed.')


## Research documentation: reduction summary

A dated reduction summary is saved beside the fairness results.


In [ ]:
from pathlib import Path
analysis_folder=Path(scenario_info['results_dir'])/'analysis'
analysis_folder.mkdir(parents=True,exist_ok=True)
reduction_brief=analysis_folder/'reduction_notes.md'
stages=reduction_audit['stages']
lines=[f'# Reduction evidence — {DATE}', '',
       f'Source resolution: {SOURCE_DT:g} h; analyzed resolution: {TARGET_DT:g} h; horizon: 24 h.',
       'Temporal aggregation is a modeling approximation, distinct from certified redundancy elimination.',
       '', '| Stage | Decisions | Shared constraints | Local inequalities |',
       '|---|---:|---:|---:|']
for st in stages:
    lines.append(f"| {st['stage']} | {st['decisions']} | {st['shared_total']} | {st['local_rows']} |")
lines.extend(['',f"EV mixed availability slots: {reduction_audit['mixed_ev_availability_slots']}",
     f"LP pruning enabled: {reduction_audit['lp_pruned']}",
     f"Model snapshot: {Path(scenario_info['model_path']).resolve()}",
     f"Reduction certificates: {Path(scenario_info['audit_path']).resolve()}",
     '', 'The report does not assert equivalence at 30-minute resolution without a separate cross-resolution check.'])
reduction_brief.write_text('\n'.join(lines)+'\n',encoding='utf8')
assert '# Reduction evidence' in reduction_brief.read_text(encoding='utf8')
print('Saved reduction notes:',reduction_brief.resolve())

## Next step

Run `strategic_GNE_selection_demo.ipynb`. Choose any `DATE` at the top **of that notebook** and run all cells. It does **not** require matching dates to be entered in this notebook: the demo builds and saves a reduced model automatically when necessary. All outputs are stored in date-specific paths.
